# features-sample
Generated from `code\business_entity_resolution\src\run_features.py` by `jobs/build_notebook.py`. Edit the script, not this notebook.

In [ ]:
!pip install -q polars==1.44.2 rapidfuzz==3.14.6 anyascii==0.3.3 scipy==1.18.1 scikit-learn==1.9.1 sparse-dot-topn==1.2.0

In [ ]:
import os
os.makedirs('/tmp/src', exist_ok=True)
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, dirs, files)

In [ ]:
%%writefile /tmp/src/features.py
"""Pair features for the main model (stream A -> B, table 6 in the README).

One row per candidate pair (s1, cand), in the input order, with columns s1, cand, f_* (float32, NaN = missing).
No label, country, source or language column: France exists only in test, so every feature is country-agnostic.
The record's country is used only to pick the legal-word list, exactly as normalisation does.

Inputs:
- pairs: the candidate table (table 4/5): s1, cand, score_*/rank_* per blocking method, best_* (optional; else
  computed over the given list), p_u50 (optional). Missing blocking columns give all-NaN features, so the schema
  is the same for every split.
- s1_rec / pool_rec: normalised records (table 1) of the S1 and S2/S3 side, passed through prepare_records().
- stats: PoolStats from the WHOLE side (all S1 and all S2+S3 records of train, or of test), no labels.

Groups:
1. name: similarities on full_name, core_name and `nolegal` (legal words removed at ANY position, the agreed fix
   for "Willow LLC Center"); first/last token, extra tokens, IDF-weighted overlap, rarest differing token.
2. legal form: agreement of the edge legal form (`legal`) and of legal words found anywhere in the name.
3. address: fuzzy similarities, city / state / dÃ©partement agreement, city found in the other address,
   number tokens (shared, conflicting, first/last, long 5+ digit tokens such as postcodes).
4. blocking: scores, ranks, gaps to the S1's best, number of methods, pruner score and list position.
5. context: name frequency in the side's files; the pair's value minus the best OTHER candidate of the same S1;
   duplicates of the candidate's name in the S1's list.
"""
from __future__ import annotations

import math
from dataclasses import dataclass

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler

from normalise import _legal_for

FEATURE_VERSION = "feat-v1"
METHODS = ("name", "name_city", "name_addr", "reverse", "rare")
FORWARD = ("name", "name_city", "name_addr")
REC_COLS = ("entity_id", "country", "full_name", "core_name", "legal", "addr_norm", "city", "state", "dept", "numbers")
TRAILING_ONLY = {"pra", "li"}  # Indian legal spellings that are legal only as a trailing token (normalisation rule)
NAME_VARIANTS = ("full", "core", "nolegal")
LONG_NUM = 5  # digit tokens this long are postcodes (US ZIP, Indian PIN, French CP) or long house numbers
CONTEXT = ("f_name_core_tsr", "f_name_core_ratio", "f_name_idf_jacc", "f_addr_tsr", "f_addr_c3_jacc", "f_num_jacc")


# ---------------------------------------------------------------------------- records and side statistics

def legal_words(country: str | None) -> dict[str, str]:
    """Legal word -> canonical form for this country, for removal at any position (trailing-only forms excluded)."""
    table = _legal_for((country or "").strip().lower())
    return {w: c for w, c in table.items() if w not in TRAILING_ONLY}


def prepare_records(rec: pl.DataFrame) -> pl.DataFrame:
    """Normalised records -> the columns used here, nulls filled, plus `nolegal` (full_name without legal words at
    any position; falls back to core_name) and `legal_any` (sorted canonical legal words found anywhere)."""
    rec = rec.select(REC_COLS).with_columns(
        *[pl.col(c).fill_null("") for c in ("country", "full_name", "core_name", "legal", "addr_norm")],
        pl.col("numbers").fill_null([]),
    )
    key = pl.col("country").str.to_lowercase().str.strip_chars()
    parts = []
    for k, g in rec.with_columns(key.alias("_k")).group_by("_k", maintain_order=True):
        table = legal_words(k[0])
        words = list(table)
        toks = pl.col("full_name").str.split(" ")
        parts.append(g.with_columns(
            toks.list.eval(pl.element().filter(~pl.element().is_in(words))).list.join(" ").alias("nolegal"),
            toks.list.eval(pl.element().filter(pl.element().is_in(words)).replace_strict(table, default=None))
            .list.unique().list.sort().list.join(" ").alias("legal_any"),
        ))
    out = pl.concat(parts) if parts else rec.with_columns(pl.lit("").alias("nolegal"), pl.lit("").alias("legal_any"), pl.lit("").alias("_k"))
    return out.with_columns(pl.when(pl.col("nolegal") == "").then(pl.col("core_name")).otherwise(pl.col("nolegal")).alias("nolegal")).drop("_k")


@dataclass
class PoolStats:
    """Side statistics (no labels): token document frequency of pool core names, and name counts."""
    n_pool: int
    token_df: dict[str, int]
    pool_name_n: pl.DataFrame  # core_name, n: S2+S3 records with this core_name
    s1_name_n: pl.DataFrame  # core_name, n: S1 records with this core_name


def pool_stats(s1_names: pl.Series, pool_names: pl.Series) -> PoolStats:
    """s1_names / pool_names: core_name of ALL S1 / ALL S2+S3 records of one side (train or test)."""
    pool_names = pool_names.fill_null("")
    df = (pool_names.str.split(" ").list.unique().explode(empty_as_null=False).drop_nulls().to_frame("t")
          .filter(pl.col("t") != "").group_by("t").len())
    count = lambda s: s.fill_null("").to_frame("core_name").group_by("core_name").len("n")
    return PoolStats(len(pool_names), dict(zip(df["t"].to_list(), df["len"].to_list())), count(pool_names), count(s1_names))


# ---------------------------------------------------------------------------- scalar helpers

def _f32(name: str, a) -> pl.Series:
    return pl.Series(name, np.asarray(a, dtype=np.float32)).fill_nan(None)


def _grams(s: str, n: int) -> set[str]:
    if n == 0:
        return set(s.split())
    s = f" {s} "
    return {s[i:i + n] for i in range(len(s) - n + 1)} if len(s) >= n else {s}


def _jaccard(a: list[str], b: list[str], n: int = 0) -> np.ndarray:
    """Token (n=0) or character n-gram Jaccard; NaN when either side is empty."""
    out = np.full(len(a), np.nan, np.float32)
    for i, (s, t) in enumerate(zip(a, b)):
        if s and t:
            A, B = _grams(s, n), _grams(t, n)
            out[i] = len(A & B) / len(A | B)
    return out


def _fuzzy(scorer, a: list[str], b: list[str]) -> np.ndarray:
    """rapidfuzz scorer in 0..1 over aligned pairs; NaN when either side is empty."""
    s = process.cpdist(a, b, scorer=scorer, workers=-1).astype(np.float32)
    if scorer is not JaroWinkler.normalized_similarity:
        s /= 100.0
    empty = (np.array([len(x) for x in a]) == 0) | (np.array([len(x) for x in b]) == 0)
    s[empty] = np.nan
    return s


# ---------------------------------------------------------------------------- feature groups (row level)

def _name_features(q: dict, p: dict, stats: PoolStats) -> list[pl.Series]:
    out = []
    for v, col in zip(NAME_VARIANTS, ("full_name", "core_name", "nolegal")):
        a, b = q[col], p[col]
        out += [_f32(f"f_name_{v}_ratio", _fuzzy(fuzz.ratio, a, b)),
                _f32(f"f_name_{v}_tsr", _fuzzy(fuzz.token_set_ratio, a, b)),
                _f32(f"f_name_{v}_tsort", _fuzzy(fuzz.token_sort_ratio, a, b)),
                _f32(f"f_name_{v}_jw", _fuzzy(JaroWinkler.normalized_similarity, a, b)),
                _f32(f"f_name_{v}_tok_jacc", _jaccard(a, b, 0)),
                _f32(f"f_name_{v}_c3_jacc", _jaccard(a, b, 3))]
    a, b = q["core_name"], p["core_name"]
    out.append(_f32("f_name_core_partial", _fuzzy(fuzz.partial_ratio, a, b)))
    n = len(a)
    first, last, only_q, only_p, len_diff, contains, idf_j, idf_max = (np.full(n, np.nan, np.float32) for _ in range(8))
    w_unknown = math.log(stats.n_pool + 1)
    idf = lambda t: math.log((stats.n_pool + 1) / (stats.token_df.get(t, 0) + 1))
    for i, (s, t) in enumerate(zip(a, b)):
        if not (s and t):
            continue
        ts, tt = s.split(), t.split()
        A, B = set(ts), set(tt)
        first[i], last[i] = ts[0] == tt[0], ts[-1] == tt[-1]
        only_q[i], only_p[i] = len(A - B), len(B - A)
        len_diff[i] = abs(len(s) - len(t))
        contains[i] = f" {s} " in f" {t} " or f" {t} " in f" {s} "
        w = {x: idf(x) for x in A | B}
        union = sum(w.values())
        idf_j[i] = sum(w[x] for x in A & B) / union if union > 0 else 1.0
        diff = [w[x] for x in A ^ B]
        idf_max[i] = max(diff) / w_unknown if diff else 0.0
    out += [_f32("f_name_first_eq", first), _f32("f_name_last_eq", last), _f32("f_name_only_s1", only_q),
            _f32("f_name_only_cand", only_p), _f32("f_name_len_diff", len_diff), _f32("f_name_contains", contains),
            _f32("f_name_idf_jacc", idf_j), _f32("f_name_idf_max_diff", idf_max)]
    return out


def _legal_features(q: dict, p: dict) -> list[pl.Series]:
    out = []
    for tag, col in (("", "legal"), ("_any", "legal_any")):
        a, b = q[col], p[col]
        ea, eb = np.array([not x for x in a]), np.array([not x for x in b])
        same = np.array([x == y for x, y in zip(a, b)], np.float32)
        same[ea | eb] = np.nan
        out += [_f32(f"f_legal{tag}_both_missing", ea & eb), _f32(f"f_legal{tag}_one_missing", ea ^ eb),
                _f32(f"f_legal{tag}_same", same)]
    out.append(_f32("f_legal_tok_jacc", _jaccard(q["legal"], p["legal"], 0)))
    return out


def _eq(a: list, b: list) -> np.ndarray:
    """1/0 equality; NaN when either value is missing (null or empty)."""
    return np.array([np.nan if not x or not y else float(x == y) for x, y in zip(a, b)], np.float32)


def _address_features(q: dict, p: dict) -> list[pl.Series]:
    a = [s.replace(",", " ").replace("  ", " ") for s in q["addr_norm"]]
    b = [s.replace(",", " ").replace("  ", " ") for s in p["addr_norm"]]
    qc, pc = q["city"], p["city"]
    city_fz = _fuzzy(fuzz.ratio, [c or "" for c in qc], [c or "" for c in pc])
    city_in = np.full(len(a), np.nan, np.float32)
    for i, (c1, c2, s, t) in enumerate(zip(qc, pc, a, b)):
        hits = [f" {c} " in f" {addr} " for c, addr in ((c1, t), (c2, s)) if c and addr]
        if hits:
            city_in[i] = any(hits)
    out = [_f32("f_addr_missing", [not s or not t for s, t in zip(a, b)]),
           _f32("f_addr_ratio", _fuzzy(fuzz.ratio, a, b)),
           _f32("f_addr_tsr", _fuzzy(fuzz.token_set_ratio, a, b)),
           _f32("f_addr_tsort", _fuzzy(fuzz.token_sort_ratio, a, b)),
           _f32("f_addr_partial", _fuzzy(fuzz.partial_ratio, a, b)),
           _f32("f_addr_tok_jacc", _jaccard(a, b, 0)),
           _f32("f_addr_c3_jacc", _jaccard(a, b, 3)),
           _f32("f_city_same", _eq(qc, pc)), _f32("f_city_ratio", city_fz), _f32("f_city_in_addr", city_in),
           _f32("f_state_same", _eq(q["state"], p["state"])), _f32("f_dept_same", _eq(q["dept"], p["dept"]))]
    n = len(a)
    cols = {k: np.full(n, np.nan, np.float32) for k in
            ("n_s1", "n_cand", "shared", "only_s1", "only_cand", "jacc", "first_eq", "last_eq", "long_shared", "long_conflict")}
    for i, (x, y) in enumerate(zip(q["numbers"], p["numbers"])):
        x, y = list(x or []), list(y or [])
        A, B = set(x), set(y)
        cols["n_s1"][i], cols["n_cand"][i] = len(x), len(y)
        if not (x and y):
            continue
        cols["shared"][i], cols["only_s1"][i], cols["only_cand"][i] = len(A & B), len(A - B), len(B - A)
        cols["jacc"][i] = len(A & B) / len(A | B)
        cols["first_eq"][i], cols["last_eq"][i] = x[0] == y[0], x[-1] == y[-1]
        LA, LB = {t for t in A if len(t) >= LONG_NUM}, {t for t in B if len(t) >= LONG_NUM}
        if LA and LB:
            cols["long_shared"][i] = len(LA & LB)
            cols["long_conflict"][i] = not (LA & LB)
    return out + [_f32(f"f_num_{k}", v) for k, v in cols.items()]


def _blocking_features(x: pl.DataFrame) -> list[pl.Expr]:
    """From the candidate table; absent columns give all-NaN features so every split has the same schema."""
    has = set(x.columns)
    col = lambda c: pl.col(c).cast(pl.Float32) if c in has else pl.lit(None, dtype=pl.Float32)
    best = lambda m: col(f"best_{m}") if f"best_{m}" in has else col(f"score_{m}").max().over("s1")
    out = [*[col(f"score_{m}").alias(f"f_score_{m}") for m in METHODS],
           *[col(f"rank_{m}").alias(f"f_rank_{m}") for m in METHODS],
           *[(best(m) - col(f"score_{m}")).alias(f"f_gap_{m}") for m in FORWARD],
           (col("best_reverse") - col("score_reverse")).alias("f_gap_reverse"),
           pl.sum_horizontal(*[col(f"rank_{m}").is_not_null() for m in METHODS]).cast(pl.Float32).alias("f_n_methods"),
           col("p_u50").alias("f_p_u50"),
           (col("p_u50").max().over("s1") - col("p_u50")).alias("f_p_gap"),
           col("p_u50").rank("ordinal", descending=True).over("s1").cast(pl.Float32).alias("f_list_rank"),
           pl.len().over("s1").cast(pl.Float32).alias("f_list_size")]
    return out


# ---------------------------------------------------------------------------- assembly

def row_features(pairs: pl.DataFrame, s1_rec: pl.DataFrame, pool_rec: pl.DataFrame, stats: PoolStats) -> pl.DataFrame:
    """Per-pair features that need only the pair itself (safe to compute in row chunks). Keeps pairs' order and
    carries q_core / p_core for the context step."""
    cols = ("full_name", "core_name", "nolegal", "legal", "legal_any", "addr_norm", "city", "state", "dept", "numbers")
    x = (pairs.select("s1", "cand").with_row_index("_i")
         .join(s1_rec.select(pl.col("entity_id").alias("s1"), *[pl.col(c).alias(f"q_{c}") for c in cols]), on="s1", how="left")
         .join(pool_rec.select(pl.col("entity_id").alias("cand"), *[pl.col(c).alias(f"p_{c}") for c in cols]), on="cand", how="left")
         .sort("_i"))
    missing = x.filter(pl.col("q_core_name").is_null() | pl.col("p_core_name").is_null()).height
    if missing:
        raise ValueError(f"{missing} pairs have no normalised record for s1 or cand")
    q = {c: x[f"q_{c}"].to_list() for c in cols}
    p = {c: x[f"p_{c}"].to_list() for c in cols}
    feats = [*_name_features(q, p, stats), *_legal_features(q, p), *_address_features(q, p)]
    return (x.select("s1", "cand", pl.col("q_core_name").alias("q_core"), pl.col("p_core_name").alias("p_core"))
            .with_columns(*feats)
            .join(stats.s1_name_n.rename({"core_name": "q_core", "n": "_s1n"}), on="q_core", how="left", maintain_order="left")
            .join(stats.pool_name_n.rename({"core_name": "q_core", "n": "_pq"}), on="q_core", how="left", maintain_order="left")
            .join(stats.pool_name_n.rename({"core_name": "p_core", "n": "_pp"}), on="p_core", how="left", maintain_order="left")
            .with_columns(*[pl.col(c).fill_null(0).cast(pl.Float32).log1p().alias(f)
                            for c, f in (("_s1n", "f_freq_s1_name_in_s1"), ("_pq", "f_freq_s1_name_in_pool"),
                                         ("_pp", "f_freq_cand_name_in_pool"))])
            .drop("_s1n", "_pq", "_pp"))


def context_features(x: pl.DataFrame) -> pl.DataFrame:
    """Needs every candidate of an S1 in x: value minus the best OTHER candidate of the same S1 (null when the list
    has one candidate), and how many other candidates share this candidate's core name."""
    x = x.with_row_index("_i")
    top = x.group_by("s1").agg(*[pl.col(c).drop_nulls().top_k(2).alias(f"_top_{c}") for c in CONTEXT])
    x = x.join(top, on="s1", how="left").sort("_i")
    other = lambda c: (pl.when(pl.col(c) == pl.col(f"_top_{c}").list.get(0, null_on_oob=True))
                       .then(pl.col(f"_top_{c}").list.get(1, null_on_oob=True))
                       .otherwise(pl.col(f"_top_{c}").list.get(0, null_on_oob=True)))
    return x.with_columns(
        *[(pl.col(c) - other(c)).alias(f"f_ctx_{c[2:]}") for c in CONTEXT],
        (pl.len().over("s1", "p_core") - 1).cast(pl.Float32).alias("f_ctx_same_name_in_list"),
        (pl.col("f_name_core_tsr") >= 1.0).sum().over("s1").cast(pl.Float32).alias("f_ctx_n_name_exact"),
    ).drop("_i", *[f"_top_{c}" for c in CONTEXT])


def build(pairs: pl.DataFrame, s1_rec: pl.DataFrame, pool_rec: pl.DataFrame, stats: PoolStats,
          chunk: int = 1_000_000) -> pl.DataFrame:
    """Table 6 for one split: s1, cand, f_* (float32, NaN = missing), rows in the order of `pairs`.
    `pairs` must hold every candidate of each S1 it contains (context and list features are per S1)."""
    dup = pairs.select("s1", "cand").is_duplicated().sum()
    if dup:
        raise ValueError(f"{dup} duplicated (s1, cand) rows")
    blocking = pairs.select("s1", "cand", *_blocking_features(pairs))
    rows = pl.concat([row_features(pairs[lo:lo + chunk], s1_rec, pool_rec, stats) for lo in range(0, pairs.height, chunk)]
                     or [row_features(pairs.clear(), s1_rec, pool_rec, stats)])
    x = context_features(rows).drop("q_core", "p_core")
    assert x.select("s1", "cand").equals(blocking.select("s1", "cand")), "row order changed"
    x = x.hstack(blocking.drop("s1", "cand"))
    f = [c for c in x.columns if c.startswith("f_")]
    return x.select("s1", "cand", *[pl.col(c).cast(pl.Float32).fill_null(float("nan")) for c in f])


def feature_names(x: pl.DataFrame) -> list[str]:
    return [c for c in x.columns if c.startswith("f_")]


In [ ]:
%%writefile /tmp/src/normalise.py
"""Shared normalisation for names and addresses (used by blocking, features and inference).

Deterministic and vectorised with polars; the only per-string Python call is anyascii
transliteration, applied once per distinct non-ASCII string.

Open-set countries: rules are looked up by the lower-cased country label. Country tables only ADD
cleaning for known countries; any other label (new in test, empty, made up) gets the generic rules
(transliteration, cleaning, generic legal suffixes and abbreviations, number tokens, city guess).

Names   -> full_name (legal suffix canonicalised), core_name (legal suffix removed), legal (the suffix)
Address -> addr_norm, city, state, dept (French departement), numbers (digit tokens, leading zeros stripped)
Fallbacks: core_name falls back to full_name; text fields are empty only when the raw value has no
letters or digits. city / state / dept are null when not found. normalise_records() keeps the raw columns.

What is learned from data (see docs/normalisation.md):
- state aliases: from TRAIN true pairs only (labels), excluding validation S1; saved to a versioned file.
- city frequency vocabulary: from train + test records (no labels), per country.
"""
from __future__ import annotations

import json
import re
from collections.abc import Mapping
from pathlib import Path

import polars as pl
from anyascii import anyascii

# Bump whenever a rule or list changes; stored in every output so cached artifacts can be checked.
NORM_VERSION = "norm-v3"  # v2: international legal forms only for France / unknown countries; v3: cie, Malayalam/Gurmukhi ltd/pvt, no dangling "and"

# ---------------------------------------------------------------------------- basic cleaning


def transliterate(s: pl.Series) -> pl.Series:
    """anyascii for non-ASCII strings (Devanagari, Tamil, accents, ...); ASCII strings untouched."""
    s = s.cast(pl.String).fill_null("")
    mask = s.str.contains(r"[^\x00-\x7F]")
    uniq = s.filter(mask).unique().to_list()
    if not uniq:
        return s
    return s.replace({u: anyascii(u) for u in uniq})


def _clean(e: pl.Expr, drop_dots: bool) -> pl.Expr:
    """Lowercase, & -> and, punctuation -> space, collapse whitespace. For names, dots and
    apostrophes are deleted (l.l.c. -> llc, nash's -> nashs); for addresses they become spaces
    (no.330 -> no 330)."""
    e = e.str.to_lowercase().str.replace_all("&", " and ")
    if drop_dots:
        e = e.str.replace_all(r"[.'`]", "")
    return e.str.replace_all(r"[^a-z0-9]+", " ").str.strip_chars()


def clean_text(text: str, drop_dots: bool = False) -> str:
    """Scalar version of transliterate + _clean (for building lookup tables and tests)."""
    s = anyascii(text).lower().replace("&", " and ")
    if drop_dots:
        s = re.sub(r"[.'`]", "", s)
    return " ".join(re.findall(r"[a-z0-9]+", s))


def _key(country: str | None) -> str:
    return (country or "").strip().lower()


def _ckey(countries: pl.Series) -> pl.Series:
    return countries.cast(pl.String).fill_null("").str.to_lowercase().str.strip_chars()


# ---------------------------------------------------------------------------- name rules (language knowledge)

# Legal suffixes: token -> canonical token. Only a trailing run (or, for LEGAL_LEADING, a leading run)
# of these is treated as a legal form; the same words mid-name are kept.
LEGAL_GENERAL = {  # all countries
    "inc": "inc", "incorporated": "inc", "incorporation": "inc",
    "llc": "llc", "llp": "llp", "lp": "lp", "plc": "plc", "pllc": "pllc", "pc": "pc", "psc": "psc",
    "corp": "corp", "corporation": "corp", "co": "co", "company": "co", "cos": "co",
    "ltd": "ltd", "limited": "ltd", "pvt": "pvt", "private": "pvt", "opc": "opc",
}
# Continental-European forms. Applied to France and to every country WITHOUT its own table (open set),
# but not to the US or India, where "spa" / "sas" / "sl" are ordinary words ("Ramey Spa Inc", "Sas Nagar").
LEGAL_INTERNATIONAL = {
    "gmbh": "gmbh", "bv": "bv", "srl": "srl", "spa": "spa", "sl": "sl", "slu": "slu",
    "sarl": "sarl", "sas": "sas", "sasu": "sasu", "eurl": "eurl", "selarl": "selarl", "eirl": "eirl", "scop": "scop",
    "cie": "co",  # "& Cie" = "& Co"
}
LEGAL_BY_COUNTRY = {
    "us": {},
    "india": {
        # transliterations seen in S2/S3 (Devanagari / Tamil / Kannada / Odia -> anyascii), e.g. "pra li" = "pvt ltd"
        "praivet": "pvt", "praibhet": "pvt", "piraivet": "pvt", "praivett": "pvt", "prayvet": "pvt", "pra": "pvt",
        "praivrr": "pvt",  # Malayalam
        "limitet": "ltd", "limited": "ltd", "limitedd": "ltd", "li": "ltd", "elelpi": "llp", "ellpi": "llp",
        "limirrd": "ltd",  # Malayalam
        "limtid": "ltd",  # Gurmukhi
    },
    "france": {"sa": "sa", "sci": "sci", "snc": "snc", "ei": "ei", "sca": "sca", "scp": "scp", "gie": "gie", "scm": "scm"},
}
# Legal forms that also appear as a LEADING token after word reordering ("LLC Value Electronics").
# Only unambiguous abbreviations: never words like "company" or "limited", or short forms like "co"/"sa".
LEGAL_LEADING = {"inc", "llc", "llp", "ltd", "pvt", "corp", "plc", "pllc"}
LEGAL_LEADING_INTERNATIONAL = {"gmbh", "sarl", "sas", "sasu", "eurl", "snc"}
KNOWN_ENGLISH_ONLY = {"us", "india"}  # countries whose tables replace the international forms


def _legal_for(country: str) -> dict[str, str]:
    extra = {} if country in KNOWN_ENGLISH_ONLY else LEGAL_INTERNATIONAL
    return {**LEGAL_GENERAL, **extra, **LEGAL_BY_COUNTRY.get(country, {})}


def _leading_for(country: str) -> set[str]:
    return LEGAL_LEADING if country in KNOWN_ENGLISH_ONLY else LEGAL_LEADING | LEGAL_LEADING_INTERNATIONAL
# multi-token spellings, collapsed before suffix detection (anchored at the end of the name)
LEGAL_MULTI = [(r" l l c$", " llc"), (r" l l p$", " llp"), (r" p l l c$", " pllc"), (r" l p$", " lp"), (r" p c$", " pc")]

# Honorific prefixes (repeated prefixes are all removed): India only, where they are noise added to names.
HONORIFICS_BY_COUNTRY = {
    "india": ["m s", "ms", "messrs", "smt", "shrimati", "shri", "shree", "sri", "sree", "kumari", "km", "mr", "mrs", "dr"],
}

# Word abbreviations (token -> expansion)
NAME_ABBREV_GENERAL = {
    "intl": "international", "mfg": "manufacturing", "mfrs": "manufacturers",
    "svc": "services", "svcs": "services", "srvcs": "services", "mgmt": "management", "mgt": "management",
    "assn": "association", "assoc": "associates", "dept": "department", "natl": "national",
    "govt": "government", "univ": "university", "hosp": "hospital", "ctr": "center", "cntr": "center",
    "centre": "center", "bros": "brothers", "engg": "engineering", "engr": "engineering", "grp": "group",
    "hldgs": "holdings", "inds": "industries", "sys": "systems", "solns": "solutions",
}
NAME_ABBREV_BY_COUNTRY = {"france": {"et": "and", "st": "saint", "ste": "sainte"}}


# ---------------------------------------------------------------------------- address rules (language knowledge)

ADDR_ABBREV_GENERAL = {
    "rd": "road", "ave": "avenue", "blvd": "boulevard", "hwy": "highway", "pkwy": "parkway",
    "bldg": "building", "apt": "apartment", "flr": "floor",
}
ADDR_ABBREV_BY_COUNTRY = {
    "us": {
        "st": "street", "str": "street", "dr": "drive", "ln": "lane", "ct": "court", "cir": "circle", "pl": "place",
        "sq": "square", "ste": "suite", "fl": "floor", "ter": "terrace", "terr": "terrace", "trl": "trail",
        "pt": "point", "mt": "mount", "ft": "fort", "hts": "heights", "rte": "route", "fwy": "freeway",
        "expy": "expressway", "cres": "crescent",
    },
    "india": {"nr": "near", "opp": "opposite", "fl": "floor", "ngr": "nagar", "bldg": "building"},
    "france": {
        "st": "saint", "ste": "sainte", "av": "avenue", "bd": "boulevard", "bld": "boulevard", "bvd": "boulevard",
        "boul": "boulevard", "rte": "route", "chem": "chemin", "pl": "place", "imp": "impasse", "fbg": "faubourg",
        "sq": "square", "all": "allee", "r": "rue",
    },
}
LANDMARK_PREFIX = r"^(near|nr|opp|opposite|behind|beside|next to|adjacent to|in front of)\b"
COUNTRY_WORDS = {"india", "bharat", "usa", "us", "united states", "united states of america", "america", "france",
                 "null", "none", "na", "n a", "nil"}
CITY_PREFIX = r"^(city of|town of|village of|township of|borough of|ville de)\s+"
CITY_SUFFIX = r"\s+(city|township|town|village|borough|cdp)$"
# A component "<postcode> <words>" or "<words> <postcode>" (4-6 digits) is a city candidate once the
# postcode is removed, unless the words look like a street (then the number is a house number) or a
# unit / box ("po box 4823", "suite 1200", "pmb 12345").
POSTCODE_EDGE = r"^\d{4,6}\s+|\s+\d{4,6}$"
STREET_WORDS = (r"\b(road|street|avenue|lane|drive|boulevard|highway|parkway|court|circle|trail|terrace|way|"
                r"rue|allee|impasse|chemin|route|quai|calle|avenida|carrer|via|viale|piazza|marg|floor|building|"
                r"po|box|pmb|unit|suite|ste|apt|apartment|room|rm|flat|plot|no|door|house|block|office|shop|sector|"
                r"ward|lot|survey|khasra|gali|lane)\b"
                r"|(strasse|gasse|weg|platz)\b")

US_STATES = {
    "al": "alabama", "ak": "alaska", "az": "arizona", "ar": "arkansas", "ca": "california", "co": "colorado",
    "ct": "connecticut", "de": "delaware", "fl": "florida", "ga": "georgia", "hi": "hawaii", "id": "idaho",
    "il": "illinois", "in": "indiana", "ia": "iowa", "ks": "kansas", "ky": "kentucky", "la": "louisiana",
    "me": "maine", "md": "maryland", "ma": "massachusetts", "mi": "michigan", "mn": "minnesota",
    "ms": "mississippi", "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada",
    "nh": "new hampshire", "nj": "new jersey", "nm": "new mexico", "ny": "new york", "nc": "north carolina",
    "nd": "north dakota", "oh": "ohio", "ok": "oklahoma", "or": "oregon", "pa": "pennsylvania",
    "ri": "rhode island", "sc": "south carolina", "sd": "south dakota", "tn": "tennessee", "tx": "texas",
    "ut": "utah", "vt": "vermont", "va": "virginia", "wa": "washington", "wv": "west virginia",
    "wi": "wisconsin", "wy": "wyoming", "dc": "district of columbia", "pr": "puerto rico", "gu": "guam",
    "vi": "virgin islands", "as": "american samoa", "mp": "northern mariana islands",
}
# canonical = full state name; value = codes and alternative spellings
INDIA_STATES = {
    "andhra pradesh": ["ap"], "arunachal pradesh": ["ar"], "assam": ["as"], "bihar": ["br"],
    "chhattisgarh": ["cg", "ct", "chattisgarh"], "goa": ["ga"], "gujarat": ["gj", "gujrat"],
    "haryana": ["hr"], "himachal pradesh": ["hp"], "jharkhand": ["jh"], "karnataka": ["ka"],
    "kerala": ["kl", "keralam"], "madhya pradesh": ["mp"], "maharashtra": ["mh"], "manipur": ["mn"],
    "meghalaya": ["ml"], "mizoram": ["mz"], "nagaland": ["nl"], "odisha": ["od", "or", "orissa"],
    "punjab": ["pb"], "rajasthan": ["rj"], "sikkim": ["sk"], "tamil nadu": ["tn", "tamilnadu"],
    "telangana": ["tg", "ts"], "tripura": ["tr"], "uttar pradesh": ["up"],
    "uttarakhand": ["uk", "ua", "ut", "uttaranchal"], "west bengal": ["wb"],
    "andaman and nicobar islands": ["an", "andaman and nicobar"], "chandigarh": ["ch"],
    "dadra and nagar haveli and daman and diu": ["dn", "dd", "dadra and nagar haveli", "daman and diu"],
    "delhi": ["dl", "nct of delhi", "national capital territory of delhi"], "jammu and kashmir": ["jk"],
    "ladakh": ["la"], "lakshadweep": ["ld"], "puducherry": ["py", "pondicherry"],
}
# canonical = current region; value = old regions / alternative spellings
FRANCE_REGIONS = {
    "auvergne rhone alpes": ["auvergne", "rhone alpes"], "bourgogne franche comte": ["bourgogne", "franche comte"],
    "bretagne": ["brittany"], "centre val de loire": ["centre"], "corse": ["corsica"],
    "grand est": ["alsace", "lorraine", "champagne ardenne"],
    "hauts de france": ["nord pas de calais", "picardie"], "ile de france": [],
    "normandie": ["basse normandie", "haute normandie", "normandy"],
    "nouvelle aquitaine": ["aquitaine", "limousin", "poitou charentes"],
    "occitanie": ["languedoc roussillon", "midi pyrenees"], "pays de la loire": [],
    "provence alpes cote d azur": ["paca", "provence alpes cote dazur"],
    "guadeloupe": [], "martinique": [], "guyane": ["french guiana"], "la reunion": ["reunion"], "mayotte": [],
}
# Departements (Paris omitted: as an address component it is the city)
FRANCE_DEPTS = [
    "ain", "aisne", "allier", "alpes de haute provence", "hautes alpes", "alpes maritimes", "ardeche", "ardennes",
    "ariege", "aube", "aude", "aveyron", "bouches du rhone", "calvados", "cantal", "charente", "charente maritime",
    "cher", "correze", "corse du sud", "haute corse", "cote d or", "cotes d armor", "creuse", "dordogne", "doubs",
    "drome", "eure", "eure et loir", "finistere", "gard", "haute garonne", "gers", "gironde", "herault",
    "ille et vilaine", "indre", "indre et loire", "isere", "jura", "landes", "loir et cher", "loire",
    "haute loire", "loire atlantique", "loiret", "lot", "lot et garonne", "lozere", "maine et loire", "manche",
    "marne", "haute marne", "mayenne", "meurthe et moselle", "meuse", "morbihan", "moselle", "nievre", "nord",
    "oise", "orne", "pas de calais", "puy de dome", "pyrenees atlantiques", "hautes pyrenees",
    "pyrenees orientales", "bas rhin", "haut rhin", "rhone", "haute saone", "saone et loire", "sarthe",
    "savoie", "haute savoie", "seine maritime", "seine et marne", "yvelines", "deux sevres", "somme", "tarn",
    "tarn et garonne", "var", "vaucluse", "vendee", "vienne", "haute vienne", "vosges", "yonne",
    "territoire de belfort", "essonne", "hauts de seine", "seine saint denis", "val de marne", "val d oise",
]


def _state_table(country: str) -> dict[str, str]:
    """Hand-written variant -> canonical state/region, keys cleaned like address components.
    Empty for any country without a table (open set)."""
    t: dict[str, str] = {}
    if country == "us":
        for code, name in US_STATES.items():
            t[code] = code
            t[clean_text(name)] = code
    elif country == "india":
        for name, alts in INDIA_STATES.items():
            t[clean_text(name)] = name
            for a in alts:
                t[clean_text(a)] = name
    elif country == "france":
        for name, alts in FRANCE_REGIONS.items():
            t[clean_text(name)] = name
            for a in alts:
                t[clean_text(a)] = name
    return t


def _dept_table(country: str) -> dict[str, str]:
    return {clean_text(d): clean_text(d) for d in FRANCE_DEPTS} if country == "france" else {}


# ---------------------------------------------------------------------------- normaliser


class Normaliser:
    """Holds the lookup tables. `state_aliases` ({country: {variant: canonical}}) are learned from
    training pairs with learn_state_aliases() and added on top of the hand-written tables."""

    def __init__(self, state_aliases: Mapping[str, Mapping[str, str]] | None = None, version: str = NORM_VERSION):
        self.state_aliases = {_key(c): dict(m) for c, m in (state_aliases or {}).items()}
        self.version = version

    @classmethod
    def load(cls, path: str | Path | None) -> "Normaliser":
        """Load aliases saved by save_state_aliases(); a missing path gives the hand-written tables only."""
        if not path or not Path(path).exists():
            return cls()
        data = json.loads(Path(path).read_text())
        if "aliases" in data:  # versioned file
            if data.get("norm_version") != NORM_VERSION:
                raise ValueError(f"{path} was learned with {data.get('norm_version')}, code is {NORM_VERSION}")
            return cls(data["aliases"])
        return cls(data)

    def states(self, country: str) -> dict[str, str]:
        return {**_state_table(country), **self.state_aliases.get(country, {})}

    # ------------------------------------------------------------------ names
    def names(self, names: pl.Series, countries: pl.Series) -> pl.DataFrame:
        raw = names.cast(pl.String).fill_null("")
        df = pl.DataFrame({"n": transliterate(raw), "ckey": _ckey(countries)})
        df = df.with_row_index("rid").with_columns(_clean(pl.col("n"), drop_dots=True).alias("n"))
        parts = [self._names_one(g, ckey) for (ckey,), g in df.group_by("ckey")]
        if not parts:
            return pl.DataFrame(schema={"full_name": pl.String, "core_name": pl.String, "legal": pl.String})
        return pl.concat(parts).sort("rid").drop("rid")

    def _names_one(self, g: pl.DataFrame, country: str) -> pl.DataFrame:
        e = pl.col("n")
        hon = HONORIFICS_BY_COUNTRY.get(country)
        if hon:
            stripped = e.str.replace(r"^(?:(?:" + "|".join(hon) + r")\s+)+", "")
            e = pl.when(stripped != "").then(stripped).otherwise(e)
        abbrev = {**NAME_ABBREV_GENERAL, **NAME_ABBREV_BY_COUNTRY.get(country, {})}
        e = e.str.split(" ").list.eval(pl.element().replace(abbrev)).list.join(" ")
        for pat, rep in LEGAL_MULTI:
            e = e.str.replace(pat, rep)
        legal = _legal_for(country)
        alt = "|".join(sorted(map(re.escape, legal), key=len, reverse=True))
        tail_pat = r"^(.*?)((?: (?:" + alt + r"))*)$"
        lead_pat = r"^((?:(?:" + "|".join(sorted(_leading_for(country))) + r") )+)(.+)$"
        g = g.with_columns(e.alias("_e")).with_columns(
            pl.col("_e").str.extract(lead_pat, 1).fill_null("").alias("_lead"),
            pl.coalesce(pl.col("_e").str.extract(lead_pat, 2), pl.col("_e")).alias("_rest"),
        )
        g = g.with_columns((pl.lit(" ") + pl.col("_rest")).alias("_s"))
        core = pl.col("_s").str.extract(tail_pat, 1).fill_null("").str.strip_chars()
        g = g.with_columns(
            core.alias("core_name"),
            pl.concat_str([pl.col("_lead"), pl.col("_s").str.extract(tail_pat, 2).fill_null("")], separator=" ")
            .str.replace_all(r"\s+", " ").str.strip_chars().str.split(" ")
            .list.eval(pl.element().replace(legal)).list.unique(maintain_order=True)  # "pvt ltd ltd" -> "pvt ltd"
            .list.join(" ").str.strip_chars().alias("legal"),
        )
        # "Smith & Co", "Elsa & Cie SARL", "Dupont et Cie": the connector belongs to the legal form
        g = g.with_columns(
            pl.when(pl.col("legal") != "").then(pl.col("core_name").str.replace(r"(?:\s+and)+$", ""))
            .otherwise(pl.col("core_name")).alias("core_name"))
        full = pl.concat_str([pl.col("core_name"), pl.col("legal")], separator=" ").str.strip_chars()
        g = g.with_columns(full.alias("full_name"))
        # Fallbacks: a name that is only legal tokens ("LLC", "Pvt Ltd") keeps them as its core.
        only_legal = pl.col("core_name") == ""
        return g.select(
            "rid",
            "full_name",
            pl.when(only_legal).then(pl.col("full_name")).otherwise(pl.col("core_name")).alias("core_name"),
            pl.when(only_legal).then(pl.lit("")).otherwise(pl.col("legal")).alias("legal"),
        )

    # ------------------------------------------------------------------ addresses
    def addresses(self, addrs: pl.Series, countries: pl.Series, city_vocab: pl.DataFrame | None = None) -> pl.DataFrame:
        """city_vocab: (ckey, cand, freq) from city_counts(); when None it is fitted on these addresses.
        The city is the most frequent city-like component, so reordered addresses agree."""
        raw = addrs.cast(pl.String).fill_null("")
        df = pl.DataFrame({"a": transliterate(raw), "ckey": _ckey(countries)}).with_row_index("rid")
        comps = self._components(df)
        vocab = city_vocab if city_vocab is not None else _vocab(comps)
        cities = (
            comps.filter(pl.col("kind") == "cand")
            .join(vocab, on=["ckey", "cand"], how="left")
            .group_by("rid")
            .agg(pl.col("cand").sort_by([pl.col("freq").fill_null(0), pl.col("pos")], descending=[True, True]).first().alias("city"))
        )
        per = comps.group_by("rid").agg(
            pl.col("c").sort_by("pos").str.join(", ").alias("addr_norm"),  # keeps component boundaries (idempotent)
            pl.col("state").drop_nulls().last().alias("state"),
            pl.col("dept").drop_nulls().last().alias("dept"),
        )
        out = (
            df.select("rid", "a")
            .join(per, on="rid", how="left")
            .join(cities, on="rid", how="left")
            .with_columns(
                pl.col("addr_norm").fill_null(""),
                _clean(pl.col("a"), drop_dots=False).str.extract_all(r"\d+")
                .list.eval(pl.element().str.replace(r"^0+(\d)", "$1")).list.unique(maintain_order=True).alias("numbers"),
            )
        )
        return out.sort("rid").select("addr_norm", "city", "state", "dept", "numbers")

    def _components(self, df: pl.DataFrame) -> pl.DataFrame:
        """One row per non-empty comma component: rid, ckey, pos, c (normalised), state, dept, kind, cand."""
        comps = (
            df.select("rid", "ckey", pl.col("a").str.split(",").alias("c"))
            .with_columns(pl.int_ranges(pl.col("c").list.len()).alias("pos"))
            .explode("c", "pos", empty_as_null=True)
            .with_columns(_clean(pl.col("c"), drop_dots=False).alias("c"))
            .filter(pl.col("c").is_not_null() & (pl.col("c") != ""))
        )
        parts = []
        for (ckey,), g in comps.group_by("ckey"):
            abbrev = {**ADDR_ABBREV_GENERAL, **ADDR_ABBREV_BY_COUNTRY.get(ckey, {})}
            # States first, on the cleaned component: US codes like "ct", "fl", "mt" would otherwise be
            # expanded to court / floor / mount. A state may carry a trailing postcode ("nc 27260").
            bare = pl.col("c").str.replace(r"(\s\d{5,6})+$", "")
            g = g.with_columns(
                bare.replace_strict(self.states(ckey), default=None, return_dtype=pl.String).alias("state"),
                bare.replace_strict(_dept_table(ckey), default=None, return_dtype=pl.String).alias("dept"),
            )
            expanded = pl.col("c").str.split(" ").list.eval(pl.element().replace(abbrev)).list.join(" ")
            g = g.with_columns(pl.coalesce(pl.col("state"), pl.col("dept"), expanded).alias("c"))
            parts.append(g)
        schema = {"rid": pl.UInt32, "ckey": pl.String, "c": pl.String, "pos": pl.Int64, "state": pl.String, "dept": pl.String}
        comps = pl.concat(parts, how="vertical_relaxed") if parts else pl.DataFrame(schema=schema)
        no_post = pl.col("c").str.replace_all(POSTCODE_EDGE, "").str.strip_chars()
        city_like = (no_post != "") & ~no_post.str.contains(r"\d") & ~no_post.str.contains(STREET_WORDS)
        stripped = no_post.str.replace(CITY_PREFIX, "").str.replace(CITY_SUFFIX, "")
        cand = pl.when(stripped != "").then(stripped).otherwise(no_post)
        kind = (
            pl.when(pl.col("state").is_not_null()).then(pl.lit("state"))
            .when(pl.col("dept").is_not_null()).then(pl.lit("dept"))
            .when(pl.col("c").str.contains(LANDMARK_PREFIX)).then(pl.lit("landmark"))
            .when(pl.col("c").is_in(list(COUNTRY_WORDS))).then(pl.lit("country"))
            .when(~pl.col("c").str.contains(r"\d")).then(pl.lit("cand"))
            .when(city_like).then(pl.lit("cand"))  # "10115 berlin", "paris 75001"
            .otherwise(pl.lit("number"))
        )
        return comps.with_columns(kind.alias("kind"), cand.alias("cand"))

    def city_counts(self, addrs: pl.Series, countries: pl.Series) -> pl.DataFrame:
        """(ckey, cand, freq) for one batch of addresses. Sum over batches (train + test, all sources,
        no labels) with merge_city_counts() to build the vocabulary passed to addresses()."""
        df = pl.DataFrame({"a": transliterate(addrs.cast(pl.String).fill_null("")), "ckey": _ckey(countries)}).with_row_index("rid")
        return _vocab(self._components(df))


def _vocab(comps: pl.DataFrame) -> pl.DataFrame:
    return comps.filter(pl.col("kind") == "cand").group_by("ckey", "cand").agg(pl.len().cast(pl.Int64).alias("freq"))


def merge_city_counts(parts: list[pl.DataFrame]) -> pl.DataFrame:
    return pl.concat(parts).group_by("ckey", "cand").agg(pl.col("freq").sum())


def normalise_records(df: pl.DataFrame, norm: Normaliser, city_vocab: pl.DataFrame | None = None,
                      name_col: str = "business_name", addr_col: str = "business_address",
                      country_col: str = "country") -> pl.DataFrame:
    """All original columns (raw name/address kept) + normalised name and address fields + norm_version."""
    n = norm.names(df[name_col], df[country_col])
    a = norm.addresses(df[addr_col], df[country_col], city_vocab)
    return pl.concat([df, n, a], how="horizontal_extend").with_columns(pl.lit(norm.version).alias("norm_version"))


# ---------------------------------------------------------------------------- learned state aliases


def learn_state_aliases(pairs: pl.DataFrame, min_count: int = 30, purity: float = 0.9,
                        max_s1_ratio: float = 0.02, max_city_concentration: float = 1.5,
                        max_with_state: float = 0.2) -> dict[str, dict[str, str]]:
    """Learn state spellings from TRAIN true pairs (columns: country, addr_1 (S1), addr_2 (match)).
    Uses labels: never pass pairs of validation S1 entities.

    A match-side city-like component becomes an alias of state X when
    - it co-occurs with S1 state X in >= `purity` of its >= `min_count` pairs,
    - it is (almost) never an S1 component (<= `max_s1_ratio`): keeps real cities such as "mumbai" out,
    - its pairs do not concentrate on one S1 city much more than the state's pairs do overall
      (<= `max_city_concentration` x): keeps old city names such as "bombay" / "poona" out,
    - it REPLACES the state: <= `max_with_state` of the match addresses containing it also contain a
      known state component. Keeps city abbreviations such as "atl" (written "ATL, GA") out."""
    base = Normaliser()
    ckey = _ckey(pairs["country"])
    s1c = base._components(pl.DataFrame({"a": transliterate(pairs["addr_1"]), "ckey": ckey}).with_row_index("rid"))
    s1_state = s1c.filter(pl.col("state").is_not_null()).group_by("rid").agg(pl.col("state").last().alias("s1_state"))
    s1_city = base.addresses(pairs["addr_1"], pairs["country"]).select("city").with_row_index("rid")
    s1_state = s1_state.join(s1_city, on="rid", how="left")
    s1_freq = s1c.group_by("ckey", "c").agg(pl.len().alias("n_s1"))
    mc = base._components(pl.DataFrame({"a": transliterate(pairs["addr_2"]), "ckey": ckey}).with_row_index("rid"))
    has_state = mc.group_by("rid").agg((pl.col("kind") == "state").any().alias("has_state"))
    unknown = mc.filter(pl.col("kind") == "cand").select("rid", "ckey", "c").unique().join(has_state, on="rid")
    co = unknown.join(s1_state, on="rid")
    per_c = co.group_by("ckey", "c").agg(pl.col("has_state").mean().alias("with_state_share"))
    city_share = (
        co.group_by("ckey", "c", "city").agg(pl.len().alias("nc"))
        .group_by("ckey", "c").agg((pl.col("nc").max() / pl.col("nc").sum()).alias("top_city_share"))
    )
    state_ckey = s1_state.join(s1c.select("rid", "ckey").unique("rid"), on="rid")
    state_share = (
        state_ckey.group_by("ckey", "s1_state", "city").agg(pl.len().alias("nc"))
        .group_by("ckey", "s1_state").agg((pl.col("nc").max() / pl.col("nc").sum()).alias("state_top_city_share"))
    )
    stats = (
        co.group_by("ckey", "c", "s1_state").agg(pl.len().alias("n"))
        .with_columns(pl.col("n").sum().over("ckey", "c").alias("n_c"))
        .join(s1_freq, on=["ckey", "c"], how="left")
        .join(city_share, on=["ckey", "c"], how="left")
        .join(state_share, on=["ckey", "s1_state"], how="left")
        .join(per_c, on=["ckey", "c"], how="left")
        .with_columns(pl.col("n_s1").fill_null(0))
        .filter((pl.col("n_c") >= min_count) & (pl.col("n") >= purity * pl.col("n_c"))
                & (pl.col("n_s1") <= max_s1_ratio * pl.col("n_c"))
                & (pl.col("top_city_share") <= max_city_concentration * pl.col("state_top_city_share"))
                & (pl.col("with_state_share") <= max_with_state))
    )
    out: dict[str, dict[str, str]] = {}
    for ck, c, st in stats.select("ckey", "c", "s1_state").sort("ckey", "c").iter_rows():
        out.setdefault(ck, {})[c] = st
    return out


def alias_diagnostics(pairs: pl.DataFrame, aliases: Mapping[str, Mapping[str, str]]) -> pl.DataFrame:
    """Evidence per learned alias (pair count, purity) for review."""
    ckey = _ckey(pairs["country"])
    base = Normaliser()
    s1c = base._components(pl.DataFrame({"a": transliterate(pairs["addr_1"]), "ckey": ckey}).with_row_index("rid"))
    s1_state = s1c.filter(pl.col("state").is_not_null()).group_by("rid").agg(pl.col("state").last().alias("s1_state"))
    mc = base._components(pl.DataFrame({"a": transliterate(pairs["addr_2"]), "ckey": ckey}).with_row_index("rid"))
    rows = [{"ckey": ck, "c": a, "state": s} for ck, m in aliases.items() for a, s in m.items()]
    if not rows:
        return pl.DataFrame(schema={"ckey": pl.String, "alias": pl.String, "state": pl.String, "pairs": pl.UInt32, "purity": pl.Float64})
    al = pl.DataFrame(rows)
    hits = mc.select("rid", "ckey", "c").unique().join(al, on=["ckey", "c"]).join(s1_state, on="rid", how="left")
    return (
        hits.group_by("ckey", "c", "state")
        .agg(pl.len().alias("pairs"), (pl.col("s1_state") == pl.col("state")).mean().alias("purity"))
        .rename({"c": "alias"})
        .sort("ckey", "pairs", descending=[False, True])
    )


def save_state_aliases(aliases: Mapping[str, Mapping[str, str]], path: str | Path, meta: Mapping | None = None) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    data = {"norm_version": NORM_VERSION, "meta": dict(meta or {}), "aliases": aliases}
    Path(path).write_text(json.dumps(data, indent=1, sort_keys=True, ensure_ascii=False))


In [ ]:
import sys
sys.path.insert(0, '/tmp/src')
sys.argv = ['run_features.py', '--input', '/kaggle/input', '--out-dir', '/kaggle/working', '--sample-s1', '3000']

In [ ]:
"""Build pair features (table 6) for the train and val candidate tables, with a quality report.

  python src/run_features.py --input /kaggle/input --out-dir /kaggle/working [--sample-s1 3000] [--splits train val]

Inputs (searched under --input): {split}_candidates.parquet (blocking v3), normalised/train_source{1,2,3}.parquet
(norm-v3). Both candidate tables hold TRAIN-side S1 (train / val roles of g25_split), so records and side
statistics come from the train files.
Outputs: features_{split}.parquet (s1, cand, f_*), features_report.md (list sizes, NaN share and AUC per feature;
the label is read from the candidate table and never written), features_config.json.
--sample-s1 N keeps N random S1 per split with ALL their candidates (lists stay whole), for test runs.
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.metrics import roc_auc_score

from features import FEATURE_VERSION, build, feature_names, pool_stats, prepare_records

SEED = 42
T0 = time.time()
NORM_COLS = ["entity_id", "country", "full_name", "core_name", "legal", "addr_norm", "city", "state", "dept", "numbers"]
P_NOTE = ("f_p_u50, f_p_gap and f_list_rank come from the blocking pruner, which was trained on 100k train-split S1: "
          "for those S1 they are in-sample (too confident). Train the model on other train S1 or drop these columns for them.")


def log(msg: str) -> None:
    try:
        import resource
        peak = f"peak {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:5.1f} GB"
    except ImportError:  # Windows
        peak = ""
    print(f"[{time.time() - T0:7.1f}s {peak}] {msg}", flush=True)


def find(root: Path, name: str, parent: str | None = None) -> Path:
    hits = sorted(p for p in root.rglob(name) if parent is None or p.parent.name == parent)
    if not hits:
        raise FileNotFoundError(f"{name} (parent {parent}) not found under {root}")
    return hits[0]


def md(df: pl.DataFrame) -> str:
    rows = [[f"{v:.4f}" if isinstance(v, float) else str(v) for v in r] for r in df.iter_rows()]
    return "\n".join(["| " + " | ".join(df.columns) + " |", "|" + "---|" * df.width, *["| " + " | ".join(r) + " |" for r in rows]])


def list_sizes(c: pl.DataFrame, split: str) -> dict:
    n = c.group_by("s1").len()["len"]
    d = {"split": split, "pairs": c.height, "s1": n.len(), "avg": n.mean(), "median": n.median(),
         "p95": n.quantile(0.95, "nearest"), "max": n.max()}
    if "p_u50" in c.columns:
        d["share_p_below_0.01"] = c.select((pl.col("p_u50") < 0.01).mean()).item()
    if "is_match" in c.columns:
        d["positives"] = int(c["is_match"].sum())
    return d


def auc_table(f: pl.DataFrame, y: np.ndarray) -> pl.DataFrame:
    """Per feature: NaN share, AUC on rows where it is present (0.5 = useless, <0.5 = inverse), and |AUC-0.5|."""
    rows = []
    for c in feature_names(f):
        v = f[c].to_numpy()
        ok = ~np.isnan(v)
        auc = roc_auc_score(y[ok], v[ok]) if ok.sum() and 0 < y[ok].sum() < ok.sum() else float("nan")
        rows.append({"feature": c, "nan_share": float(1 - ok.mean()), "auc": float(auc), "strength": abs(auc - 0.5)})
    return pl.DataFrame(rows).sort("strength", descending=True, nulls_last=True)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, type=Path)
    ap.add_argument("--out-dir", required=True, type=Path)
    ap.add_argument("--splits", nargs="*", default=["train", "val"])
    ap.add_argument("--sample-s1", type=int, default=0, help="0 = all S1")
    ap.add_argument("--chunk", type=int, default=1_000_000)
    a = ap.parse_args()
    a.out_dir.mkdir(parents=True, exist_ok=True)

    cands = {}
    for split in a.splits:
        path = find(a.input, f"{split}_candidates.parquet")
        c = pl.read_parquet(path)
        log(f"{split}: {path} {c.shape}\n  columns: {c.columns}")
        if "scope" in c.columns and c["scope"].n_unique() > 1:
            raise ValueError(f"{split}: several scopes {c['scope'].unique().to_list()}")
        if a.sample_s1:
            keep = c.select("s1").unique().sort("s1").sample(min(a.sample_s1, c["s1"].n_unique()), seed=SEED)
            c = c.join(keep, on="s1", how="semi")
        cands[split] = c
    sizes = [list_sizes(c, s) for s, c in cands.items()]
    log(f"list sizes: {sizes}")

    ids_s1 = pl.concat([c.select(pl.col("s1").alias("entity_id")) for c in cands.values()]).unique()
    ids_pool = pl.concat([c.select(pl.col("cand").alias("entity_id")) for c in cands.values()]).unique()
    src = {k: find(a.input, f"train_source{k}.parquet", parent="normalised") for k in (1, 2, 3)}
    log(f"normalised files: {src}")
    stats = pool_stats(pl.read_parquet(src[1], columns=["core_name"])["core_name"],
                       pl.concat([pl.read_parquet(src[k], columns=["core_name"]) for k in (2, 3)])["core_name"])
    log(f"side stats: pool {stats.n_pool:,} records, {len(stats.token_df):,} tokens")
    rec = lambda paths, ids: prepare_records(pl.concat([pl.scan_parquet(p).select(NORM_COLS).join(ids.lazy(), on="entity_id", how="semi")
                                                        for p in paths]).collect())
    s1_rec, pool_rec = rec([src[1]], ids_s1), rec([src[2], src[3]], ids_pool)
    log(f"records: s1 {s1_rec.height:,}, pool {pool_rec.height:,}")

    report = [f"# Pair features {FEATURE_VERSION}", "", f"Sample: {a.sample_s1 or 'all'} S1 per split.", "",
              "## Candidate lists", "", md(pl.DataFrame(sizes)), "", f"**Note for stream B:** {P_NOTE}", ""]
    timing = {}
    for split, c in cands.items():
        t = time.time()
        f = build(c, s1_rec, pool_rec, stats, chunk=a.chunk)
        timing[split] = {"rows": f.height, "seconds": round(time.time() - t, 1),
                         "rows_per_s": round(f.height / max(time.time() - t, 1e-9))}
        f.write_parquet(a.out_dir / f"features_{split}.parquet")
        log(f"{split}: {f.shape} in {timing[split]['seconds']} s -> features_{split}.parquet")
        if "is_match" in c.columns:
            assert f.select("s1", "cand").equals(c.select("s1", "cand"))
            tab = auc_table(f, c["is_match"].cast(pl.Int8).to_numpy())
            tab.write_csv(a.out_dir / f"features_auc_{split}.csv")
            report += [f"## {split}: NaN share and AUC per feature ({f.height:,} pairs)", "", md(tab), ""]
    report += ["## Timing", "", md(pl.DataFrame([{"split": s, **v} for s, v in timing.items()])), ""]
    (a.out_dir / "features_report.md").write_text("\n".join(report))
    (a.out_dir / "features_config.json").write_text(json.dumps(
        {"feature_version": FEATURE_VERSION, "sample_s1": a.sample_s1, "splits": a.splits, "list_sizes": sizes,
         "timing": timing, "features": feature_names(f), "note": P_NOTE}, indent=1, default=str))
    log("done")


if __name__ == "__main__":
    main()
